In [ ]:
!pip install -q openai pillow

In [ ]:
import base64
from IPython.display import display, Image, Markdown
from pprint import pprint

def print_completion(completion):
    print(f"Completion id: {completion.id}")
    print(f"Input tokens: {completion.usage.prompt_tokens}; Output tokens: {completion.usage.completion_tokens}")
    print()

    if len(completion.choices) > 1:
        raise Exception("Expected a single choice.")

    message = completion.choices[0].message
    if message is None:
        raise Exception("Expected a single message")

    if message.reasoning is not None:
        display(Markdown(f"<i>[Reasoning...]</i> <br/> <i>{message.reasoning}</i> <br/>"))

    for image in message.images:
        # format_specifier looks like `data:image/png;base64,bytes...`
        format_specifier, encoded_bytes = image["image_url"]["url"].split(",", maxsplit=1)

        format = format_specifier.split(';')[0].split("/")[1]
        data = base64.b64decode(encoded_bytes)
        display(Image(data=data, format=format))

In [ ]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get('OPENROUTER_API_KEY')
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
generate_portrait_prompt = """
                           A photorealistic portrait of a software developer. He is 24-years-old, from Italy, and wears glasses.
                           The scene is illuminated by the sunlight streaming through the open window, creating a welcoming and calm atmosphere.
                           Through the window a beautiful garden can be seen with all kinds of fruit trees and vegetables
                           The overall mood is serene and masterful. Vertical portrait orientation.
                           """

In [ ]:
generate_portrait_completion = client.chat.completions.create(
    model="openai/gpt-5-image-mini",
    messages=[{ "role":"user", "content": generate_portrait_prompt }],

    # NOTE: Some models do not support the "text" modality
    modalities=["text", "image"],

    # NOTE: Some models do not support reasoning
    reasoning_effort="low",

    extra_body={
      "image_config": { "aspect_ratio": "16:9", "image_size": "0.5K" }
    }
)

In [ ]:
print_completion(generate_portrait_completion)